# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

from dotenv import load_dotenv

In [3]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [4]:
import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel
from tavily import TavilyClient

from lib.parsers import PydanticOutputParser
from lib.vector_db import VectorStoreManager
from lib.memory import LongTermMemory, MemoryFragment
from lib.tooling import Tool

# ChromaDB setup — connects to the collection built in Part 01
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base="https://openai.vocareum.com/v1"
)
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)
print(f"Collection ready: {collection.count()} documents")

Collection ready: 15 documents


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=["documents", "metadatas"]
    )

    docs = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        docs.append(
            f"Name: {meta.get('Name')} | "
            f"Platform: {meta.get('Platform')} | "
            f"Year: {meta.get('YearOfRelease')} | "
            f"Description: {meta.get('Description', doc)}"
        )

    return docs

#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    useful: bool
    description: str


@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    judge_llm = LLM(model="gpt-4o-mini")
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved Documents:\n{retrieved_docs}"
    )

    response = judge_llm.invoke(input=prompt, response_format=EvaluationReport)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(response)

    return f"Useful: {report.useful}\nDescription: {report.description}"

#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> str:
    """
    Searches the web for up-to-date information about the game industry.
    args:
    - question: a question about game industry.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    result = client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False,
    )

    answer = result.get("answer", "")
    sources = "\n".join(
        f"- {r['title']}: {r['content'][:200]}"
        for r in result.get("results", [])[:3]
    )
    return f"Answer: {answer}\n\nSources:\n{sources}"

### Agent

In [8]:
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are UdaPlay, an AI Research Agent for the video game industry. "
        "Your job is to answer questions about video games accurately and concisely.\n\n"
        "Follow this workflow for every question:\n"
        "1. Call retrieve_game to search the internal knowledge base.\n"
        "2. Call evaluate_retrieval with the question and the retrieved docs to judge their usefulness.\n"
        "3. If useful=False, call game_web_search to find the answer on the web.\n"
        "4. Provide a clear final answer and mention whether the source was the internal database or the web."
    ),
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
)

In [9]:
queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print("="*60)

    run = agent.invoke(query)
    final_state = run.get_final_state()
    messages = final_state.get("messages", [])

    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content:
            print(f"Answer: {msg.content}")
            break


Query: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Answer: Pokémon Gold and Silver was released in 1999 for the Game Boy Color. This information is sourced from the internal database.

Query: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Answer: The first 3D platformer Mario game is Super Mario 64, 

### (Optional) Advanced

In [10]:
# Long-term memory: agent stores and recalls useful facts across conversations

db = VectorStoreManager(OPENAI_API_KEY)
ltm = LongTermMemory(db)


def build_memory_registration_tool(ltm, owner, namespace):
    def _register(content: str):
        ltm.register(MemoryFragment(content=content, owner=owner, namespace=namespace))
        return "Memory saved successfully."
    return Tool(
        func=_register,
        name="register_memory",
        description=(
            "Stores a useful fact or game-related information for future reference. "
            "args: - content: the information to remember"
        ),
    )


def build_memory_search_tool(ltm, owner, namespace):
    def _search(content: str):
        result = ltm.search(query_text=content, owner=owner, namespace=namespace, limit=3)
        if not result.fragments:
            return "No relevant memories found."
        return "Stored memories:\n" + "\n".join(f"- {f.content}" for f in result.fragments)
    return Tool(
        func=_search,
        name="search_memory",
        description=(
            "Retrieves previously stored facts or game-related information. "
            "args: - content: the query to search for relevant memories"
        ),
    )


agent_with_memory = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are UdaPlay, an AI Research Agent for the video game industry. "
        "Your job is to answer questions about video games accurately and concisely.\n\n"
        "Follow this workflow for every question:\n"
        "1. Call search_memory to check if you already know the answer.\n"
        "2. Call retrieve_game to search the internal knowledge base.\n"
        "3. Call evaluate_retrieval with the question and retrieved docs to judge their usefulness.\n"
        "4. If useful=False, call game_web_search to find the answer on the web.\n"
        "5. Call register_memory to store any new useful facts you discovered.\n"
        "6. Provide a clear final answer and mention your source."
    ),
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search,
        build_memory_registration_tool(ltm, "udaplay", "games"),
        build_memory_search_tool(ltm, "udaplay", "games"),
    ],
)

# Test agent with long-term memory
run = agent_with_memory.invoke("When Pokémon Gold and Silver was released?")
final_state = run.get_final_state()
for msg in reversed(final_state.get("messages", [])):
    if isinstance(msg, AIMessage) and msg.content:
        print(msg.content)
        break

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Pokémon Gold and Silver was released in 1999 for the Game Boy Color. These games are notable for being the second generation of Pokémon games, introducing new regions and gameplay mechanics.


In [11]:
run = agent_with_memory.invoke("Why was there a pokemon leaf green and fire red, but no equivalent for pokemon blue when pokemon blue was one of the original pokemon games?")
final_state = run.get_final_state()
for msg in reversed(final_state.get("messages", [])):
    if isinstance(msg, AIMessage) and msg.content:
        print(msg.content)
        break

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Pokémon Leaf Green and Fire Red were remakes of Pokémon Red and Blue, but there was no remake of Pokémon Blue because the developers chose Leaf Green to symbolize peace and unity, contrasting with FireRed's fiery theme. This thematic decision influenced the naming and branding of the remakes. 

Source: Game Rant.
